In [1]:
import numpy as np
import pandas as pd
from scipy.spatial import KDTree

In [2]:
pd.options.mode.chained_assignment = None 

In [3]:
# Convert your Pandas DataFrame to a Dask DataFrame
temperature_long_df = pd.DataFrame({
    'x': np.random.rand(9_000_000),  # Example data
    'y': np.random.rand(9_000_000),
    'Date': np.random.choice(pd.date_range('2024-01-01', '2024-12-31'), size=9_000_000),
    'Temperature': np.random.rand(9_000_000) * 30
})

In [4]:
def create_KDTree(df):
    """
    Precompute KD-Trees for each unique date.
    """
    # Use dictionary comprehension to create KD-Trees for each unique date
    return {
        date: KDTree(group[['x', 'y']].values) 
        for date, group in df.groupby('Date')
    }

def compute_nearest_avg(df, date, k):
    """
    Compute the average temperature for the k-nearest neighbors of each point on a given date.
    """
    
    dff = df.copy(deep=True)
    
    # Create KD-Trees for all dates
    trees = create_KDTree(dff)
    
    # Filter the DataFrame for the specific date
    date_filter_df = dff[dff['Date'] == date]
    points = date_filter_df[['x', 'y']].values  # Extract coordinates as a numpy array
    
    # Get the precomputed KD-Tree for the specific date
    tree = trees[date_filter_df.Date.iloc[0]]
    
    # Query the KD-Tree to find k-nearest neighbors
    dist, idx = tree.query(points, k=k, workers=-1)

    # Extract the temperatures of the k-nearest neighbors
    nearest_temps = dff[dff['Date'] == date].iloc[idx.flatten()]['Temperature'].values.reshape(len(points), k)
    
    # Compute the average temperature for the k-nearest neighbors
    date_filter_df[f'avg_temp_k{k}'] = np.mean(nearest_temps, axis=1)
    
    return date_filter_df


In [5]:
%%time
compute_nearest_avg(temperature_long_df,'2024-01-01',3).head()

CPU times: user 3.81 s, sys: 549 ms, total: 4.36 s
Wall time: 4.31 s


,x,y,Date,Temperature,avg_temp_k3
359,0.306166,0.043737,2024-01-01,7.214665,17.022808
960,0.729870,0.198913,2024-01-01,4.292876,6.693066
1411,0.408096,0.359791,2024-01-01,23.733861,19.591323
1412,0.082177,0.810337,2024-01-01,5.753460,15.482345
2234,0.301666,0.930329,2024-01-01,12.185820,19.001875


In [6]:
%%time
from concurrent.futures import ProcessPoolExecutor

# Wrap the parallel execution in a function for better error handling
def parallel_compute_avg(df, k):
    # List to hold results
    results = []
    
    # Use ProcessPoolExecutor for parallel processing
    with ProcessPoolExecutor(max_workers=50) as executor:
        # Submit tasks for each unique date
        futures = {
            executor.submit(compute_nearest_avg, df, date, k): date for date in df['Date'].unique()
        }
        
        # Gather results
        for future in futures:
            try:
                result = future.result()  # Wait for and retrieve the result
                results.append(result)
            except Exception as e:
                print(f"Error processing date {futures[future]}: {e}")

    # Combine all results back into a single DataFrame
    return pd.concat(results, ignore_index=True)

# Call the parallel function
results_df = parallel_compute_avg(temperature_long_df, 3)


CPU times: user 20.9 s, sys: 1min 55s, total: 2min 16s
Wall time: 3min 5s


- __Number of Cores__: 50
- __Memory__: 100GB

In [7]:
results_df.head()

,x,y,Date,Temperature,avg_temp_k3
0,0.114682,0.319938,2024-04-15,14.925713,15.824505
1,0.109974,0.099329,2024-04-15,10.283130,12.151477
2,0.206279,0.378251,2024-04-15,23.935405,11.440940
3,0.538619,0.280973,2024-04-15,5.532042,4.029649
4,0.153369,0.636818,2024-04-15,5.060008,14.952579


In [8]:
results_df.shape

(9000000, 5)